# Hyperpolarisation

Magnetic resonance sees nuclear spins through their polarisation, and at
room temperature that polarisation is tiny: about three protons in a
hundred thousand point along the field at 9.4 T. Hyperpolarisation
raises it by orders of magnitude. Dissolution DNP does this for
[1-¹³C]pyruvate, which images tumour metabolism; parahydrogen does it
chemically. Each method moves polarisation through a network of coupled
nuclear spins with pulses, fields and relaxation, which is what
`SpinSystem` simulates.

This notebook shows the thermal polarisation, a free induction decay and
its spectrum, polarisation transfer by INEPT, the spectra of parahydrogen
added at high and at low field (PASADENA and ALTADENA), the field
dependence of SABRE for ¹H and ¹⁵N, SABRE with chemical exchange,
transfer to ¹³C by field cycling, the decay of hyperpolarisation by T₁,
and T₁ from the structure of a molecule.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q
from metatron_qso import spin

## Thermal polarisation

At equilibrium a spin ½ has polarisation `P = tanh(ħγB/2k_BT)`. It grows
with the field and the gyromagnetic ratio and falls with the temperature.

In [ ]:
for iso in ["1H", "13C", "15N"]:
    row = []
    for b, t in [(1.5, 310.0), (9.4, 298.15), (3.35, 1.4)]:
        s = q.SpinSystem(b, temperature=t)
        s.add_spin(iso)
        row.append(s.polarization(s.thermal_state(), 0))
    print(f"{iso:4}", "  ".join(f"{p:+.2e}" for p in row))
print("columns: 1.5 T at 310 K (a clinical scanner), 9.4 T at 298 K, 3.35 T at 1.4 K (DNP)")

## A free induction decay

A ¹³C–¹H pair at 9.4 T with relaxation. After a 90° pulse on carbon, the
carbon signal is a doublet split by `J = 140 Hz`; T₂ sets the width of
its lines. `lines` gives the exact stick spectrum, `fid` the decay and
`spin.spectrum` its Fourier transform.

In [ ]:
sys = q.SpinSystem(9.4)
h = sys.add_spin("1H", 4.0, t1=1.0, t2=0.3)
c = sys.add_spin("13C", 60.0, t1=3.0, t2=0.05)
sys.couple(h, c, 140.0)
eq = sys.thermal_state()
rho = sys.evolve(eq, [spin.pulse("13C", 90)])
dwell = 2e-4
fid = sys.fid(rho, "13C", dwell, 2048)
freqs, values = spin.spectrum(fid, dwell)
ppm = np.array([sys.ppm("13C", f) for f in freqs])
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
t = np.arange(len(fid)) * dwell
a.plot(t * 1e3, fid.imag, lw=0.8)
a.set_xlim(0, 150)
a.set_xlabel("t (ms)")
a.set_ylabel("Im FID")
keep = (ppm > 57) & (ppm < 63)
b.plot(ppm[keep], np.abs(values[keep]))
b.invert_xaxis()
b.set_xlabel("δ (ppm)")
b.set_ylabel("|S|")
plt.tight_layout()
plt.show()
for line in sys.lines(rho, "13C"):
    print(f"{line['ppm']:8.4f} ppm  {line['frequency']:10.2f} Hz  {line['amplitude']:.3e}")

## INEPT

INEPT moves the larger proton polarisation onto carbon through their
coupling: `90°x(H) – τ – 180°(H, C) – τ – 90°y(H) 90°x(C)` with
`τ = 1/(4J)`. With its two-step phase cycle, which cancels carbon's own
magnetisation, the transferred doublet is antiphase and `P_H/P_C ≈
γ_H/γ_C ≈ 3.98` times the doublet of direct excitation.

In [ ]:
ch = q.SpinSystem(9.4)
ch.add_spin("1H", 4.0)
ch.add_spin("13C", 60.0)
ch.couple(0, 1, 140.0)
eq = ch.thermal_state()
direct = ch.lines(ch.evolve(eq, [spin.pulse("13C", 90)]), "13C")
steps = [ch.lines(ch.evolve(eq, events), "13C") for events, _ in spin.inept_cycle("1H", "13C", 140.0)]
cycled = [(a["amplitude"] - b["amplitude"]) / 2 for a, b in zip(*steps)]
for d, t in zip(direct, cycled):
    print(f"{d['ppm']:8.4f} ppm  direct {d['amplitude'].imag:+.3e}  INEPT {t.imag:+.3e}  ratio {abs(t) / abs(d['amplitude']):.4f}")

## Parahydrogen at high and at low field

Hydrogenation with parahydrogen puts two protons of the product in the
nuclear singlet. Added in the magnet (PASADENA), the singlet loses its
zero-quantum coherence and becomes `¼ − I_zA I_zB`: a 45° pulse gives two
antiphase doublets, each line `1/8` in size, far above thermal signals.
Added at low field and carried adiabatically into the magnet (ALTADENA),
the pair ends in one of `|αβ⟩` and `|βα⟩`; each multiplet then shows a
dominant line, the two of opposite sign.

In [ ]:
pair = q.SpinSystem(14.1)
pair.add_spin("1H", 1.0)
pair.add_spin("1H", 5.0)
pair.couple(0, 1, 7.0)
pasadena = pair.evolve(pair.time_average(pair.parahydrogen_state(0, 1)), [spin.pulse("1H", 45)])
zero = pair.with_field(0.0)
carried = zero.evolve(zero.parahydrogen_state(0, 1), [spin.sweep(1.0, 20.0, 20000), spin.set_field(14.1)])
altadena = pair.evolve(pair.time_average(carried), [spin.pulse("1H", 20)])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, rho, title in [(axes[0], pasadena, "PASADENA, 45°"), (axes[1], altadena, "ALTADENA, 20°")]:
    for line in pair.lines(rho, "1H"):
        ax.vlines(line["ppm"], 0, line["amplitude"].imag, lw=2)
    ax.axhline(0, color="k", lw=0.5)
    ax.invert_xaxis()
    ax.set_title(title)
    ax.set_xlabel("δ (ppm)")
axes[0].set_ylabel("Im amplitude")
plt.tight_layout()
plt.show()

## SABRE

In SABRE, parahydrogen and the substrate bind to an iridium catalyst
together. The two hydrides keep their singlet, and polarisation flows to
the substrate where the hydride singlet level anti-crosses a triplet
level of the substrate: `ν_S − ν_H = ±(J_AB − J_AS/4)`. For a substrate
proton this is at a few millitesla. For ¹⁵N (SABRE-SHEATH) the Larmor
frequencies are those of different isotopes, so the field drops to about
a tenth of a microtesla and the laboratory frame is needed. The curves
are the substrate polarisation after a long, random contact time.

In [ ]:
def transfer(sys, fields):
    out = []
    for b in fields:
        at = sys.with_field(b)
        out.append(at.polarization(at.time_average(at.parahydrogen_state(0, 1)), 2))
    return np.array(out)

proton = q.SpinSystem(0.0)
for shift in (-22.0, -22.0, 8.0):
    proton.add_spin("1H", shift)
proton.couple(0, 1, -7.0)
proton.couple(0, 2, 1.2)
nitrogen = q.SpinSystem(0.0, frame="laboratory")
nitrogen.add_spin("1H", -22.0)
nitrogen.add_spin("1H", -22.0)
nitrogen.add_spin("15N", 250.0)
nitrogen.couple(0, 1, -7.0)
nitrogen.couple(0, 2, -2.0)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
fields = np.linspace(1e-3, 12e-3, 400)
a.plot(fields * 1e3, transfer(proton, fields))
a.set_xlabel("B (mT)")
a.set_ylabel("substrate polarisation")
a.set_title("¹H substrate")
fields = np.linspace(0.02e-6, 0.4e-6, 400)
b.plot(fields * 1e6, transfer(nitrogen, fields))
b.set_xlabel("B (μT)")
b.set_title("¹⁵N substrate (SABRE-SHEATH)")
plt.tight_layout()
plt.show()

## SABRE with exchange

The time average above treats the catalyst as if the substrate stayed
bound forever. In reality it binds and leaves within tens of
milliseconds, and hydrogen is replaced as well. `spin.Exchange` follows
the complex and the free substrate together: here the free ¹⁵N builds up
polarisation over seconds, most strongly near the anti-crossing. Faster
exchange first delivers more of it (k_d = 10/s beats 2/s), then
shortens the time the substrate has to receive it on the complex, which
lowers and broadens the resonance and shifts its maximum (40/s).

In [ ]:
def sheath(field, kd):
    complex_ = q.SpinSystem(field, frame="laboratory")
    for shift, iso in ((-22.0, "1H"), (-22.0, "1H"), (250.0, "15N")):
        complex_.add_spin(iso, shift)
    complex_.couple(0, 1, -7.0)
    complex_.couple(0, 2, -2.0)
    free = q.SpinSystem(field, frame="laboratory")
    free.add_spin("15N", 310.0, t1=30.0)
    x = spin.Exchange()
    x.add_site(complex_, 1.0)
    x.add_site(free, 10.0)
    x.add_channel(0, 1, kd, carried=[(2, 0)])
    x.add_channel(1, 0, kd / 10, carried=[(0, 2)], parahydrogen=(0, 1))
    x.add_channel(0, 0, 0.5, carried=[(2, 2)], parahydrogen=(0, 1))
    return x

fields = np.linspace(0.02e-6, 0.3e-6, 60)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
for kd in (2.0, 10.0, 40.0):
    p = [sheath(f, kd).polarization(sheath(f, kd).steady_state(), 1, 0) for f in fields]
    a.plot(fields * 1e6, p, label=f"k_d = {kd:g} /s")
a.set_xlabel("B (μT)")
a.set_ylabel("free ¹⁵N polarisation (steady state)")
a.legend()
x = sheath(0.139e-6, 2.0)
times = np.linspace(0, 60, 31)
states = x.thermal_state()
build = [0.0]
for dt in np.diff(times):
    states = x.evolve(states, [spin.delay(float(dt))])
    build.append(x.polarization(states, 1, 0))
b.plot(times, build)
b.set_xlabel("t (s)")
b.set_ylabel("free ¹⁵N polarisation")
plt.tight_layout()
plt.show()

## From parahydrogen to carbon by field cycling

In PHIP the two new protons of a molecule are in the singlet; a nearby
¹³C, as in [1-¹³C]pyruvate made by side-arm hydrogenation, carries none
of it. Formed near zero field and carried slowly to a few microtesla,
the singlet order becomes ¹³C polarisation; a sudden jump moves none.

In [ ]:
three = q.SpinSystem(0.0, frame="laboratory")
three.add_spin("1H", 2.0)
three.add_spin("1H", 4.0)
three.add_spin("13C", 170.0)
three.couple(0, 1, 7.2)
three.couple(0, 2, 5.8)
three.couple(1, 2, -1.4)
start = three.time_average(three.parahydrogen_state(0, 1))
for label, events in [
    ("slowly", [spin.sweep(1e-6, 40.0, 8000), spin.sweep(5e-5, 10.0, 2000)]),
    ("in 0.1 s", [spin.sweep(5e-5, 0.1, 1000)]),
    ("at once", [spin.set_field(5e-5)]),
]:
    print(f"{label:9} 13C polarisation {three.polarization(three.evolve(start, events), 2):+.3f}")

## Hyperpolarisation decays with T₁

Hyperpolarisation is a state far from equilibrium, and relaxation brings
it back: a ¹³C polarisation of 30 % at 3 T decays with `T₁` towards the
thermal `2.5·10⁻⁶`. The simulated decay of a hyperpolarised ¹³C coupled to a
proton follows `P(t) = P_eq + (P₀ − P_eq) e^{−t/T₁}`.

In [ ]:
c13 = q.SpinSystem(3.0, temperature=310.0)
c13.add_spin("13C", 170.0, t1=60.0)
c13.add_spin("1H", 2.0, t1=1.5)
c13.couple(0, 1, 5.0)
p0 = 0.3
# P = 2⟨I_z⟩ for spin ½: adding (P₀/2)·I_z adds P₀.
rho = c13.thermal_state() + p0 / 2 * c13.spin_operator(0, "z")
times = np.linspace(0, 180, 7)
p_eq = c13.polarization(c13.thermal_state(), 0)
start = c13.polarization(rho, 0)
for t in times:
    p = c13.polarization(c13.evolve(rho, [spin.delay(float(t))]), 0)
    print(f"t = {t:5.0f} s   P = {p:.5f}   exact {p_eq + (start - p_eq) * np.exp(-t / 60.0):.5f}")

## Relaxation from first principles

How long hyperpolarisation lasts depends on the molecule. With the
positions of the nuclei, a shielding tensor and the correlation time of
tumbling, `SpinSystem` computes relaxation by Redfield theory instead of
taking T₁ as input. For a carbonyl ¹³C with one proton at 2.1 Å and a
shielding anisotropy of 140 ppm, the dipolar coupling to the proton
dominates at low field, and a deuteron in its place lengthens T₁ about
eightfold at 1.5 T. At high field the shielding anisotropy, whose rate
grows with B², takes over, and deuteration gains little. (Within this
model: the deuteron's own quadrupolar relaxation, which relaxes the
carbon through their scalar coupling at low field, and relaxation by the
solvent are not included.)

In [ ]:
def carbonyl(field, partner="1H", tau=2e-11):
    s = q.SpinSystem(field)
    s.add_spin("13C", 171.0)
    s.add_spin(partner, 2.0)
    s.set_position(0, (0.0, 0.0, 0.0))
    s.set_position(1, (2.1, 0.0, 0.0))
    s.set_csa(0, spin.axial_csa(-140.0, (0.0, 1.0, 0.0)))
    return s.with_correlation_time(tau)

def t1(s):
    return 1 / s.relaxation_rates([s.spin_operator(0, "z")])[0, 0]

fields = np.linspace(0.5, 20.0, 40)
plt.figure(figsize=(6, 3.5))
for partner in ["1H", "2H"]:
    plt.plot(fields, [t1(carbonyl(b, partner)) for b in fields], label=f"¹³C with {partner}")
plt.xlabel("B (T)")
plt.ylabel("T₁ of ¹³C (s)")
plt.legend()
plt.tight_layout()
plt.show()
for b in [1.5, 3.0, 9.4]:
    print(f"{b:4} T   T1 with 1H {t1(carbonyl(b)):6.1f} s   with 2H {t1(carbonyl(b, '2H')):6.1f} s")